In [2]:
import torch
import torchvision
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader



In [4]:
train_dataset = torchvision.datasets.CIFAR10(root='/data',
                                             train=True,
                                             download=True,
                                             transform=v2.Compose([v2.ToImage(),
                                                                   v2.ToDtype(torch.float32,scale=True),
                                                                   v2.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])]))
test_dataset = torchvision.datasets.CIFAR10(root='/data',
                                             train=False,
                                             download=True,
                                             transform=v2.Compose([v2.ToImage(),
                                                                   v2.ToDtype(torch.float32,scale=True),
                                                                   v2.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])]))


100%|██████████| 170M/170M [20:26<00:00, 139kB/s]


In [5]:
tasks=[[0,1],[2],[3],[4],[5],[6],[7],[8],[9]]

In [6]:
from torch.utils.data import Subset
task_datasets=[]

for classes in tasks:
  indices=[i for i ,(_,label) in enumerate(train_dataset) if label in classes]
  task_datasets.append(Subset(train_dataset,indices))

task_test_dataset=[]
for classes in tasks:
  indices=[i for i, (_,label) in enumerate(test_dataset) if label in classes]
  task_test_dataset.append(Subset(test_dataset,indices))

In [7]:
task1_loader=DataLoader(task_datasets[0],batch_size=10,shuffle=True)
test1_loader=DataLoader(task_test_dataset[0],batch_size=10,shuffle=True)

In [14]:
class ImageNet(nn.Module):

  def __init__(self):
    super().__init__()
    self.conv_layer=nn.Sequential(
        nn.Conv2d(3,5,5,padding=2),
        nn.BatchNorm2d(5),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(5,10, 5, padding=2),
        nn.BatchNorm2d(10),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(10,20, 5, padding=2),
        nn.BatchNorm2d(20),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(20,30, 5, padding=2),
        nn.BatchNorm2d(30),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
        )
    self.linear_layer=nn.Sequential(
        nn.Linear(30*2*2,80),
        nn.BatchNorm1d(80),
        nn.ReLU(),
        nn.Linear(80,40),
        nn.BatchNorm1d(40),
        nn.ReLU(),
        nn.Linear(40,10),
        nn.BatchNorm1d(10),
        nn.ReLU(),
        nn.Linear(10,2),
      )
  def forward(self,x):
    x=self.conv_layer(x)
    x=torch.flatten(x,1)
    x=self.linear_layer(x)
    return x

In [15]:
net=ImageNet()
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(net.parameters(),lr=0.001,momentum=0.9)

In [17]:
result_loss=0
for epoch in range(10):
  for batch,data in enumerate(task1_loader):
    image,labels=data
    output=net(image)
    optimizer.zero_grad()
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()

    result_loss+=loss.item()

    if batch%100==99:
      print(f"Epoch:{epoch+1} , Batch:{batch+1} , Loss:{result_loss/1000:.4f}")
      result_loss=0

Epoch:1 , Batch:100 , Loss:0.0423
Epoch:1 , Batch:200 , Loss:0.0389
Epoch:1 , Batch:300 , Loss:0.0407
Epoch:1 , Batch:400 , Loss:0.0377
Epoch:1 , Batch:500 , Loss:0.0352
Epoch:1 , Batch:600 , Loss:0.0340
Epoch:1 , Batch:700 , Loss:0.0310
Epoch:1 , Batch:800 , Loss:0.0329
Epoch:1 , Batch:900 , Loss:0.0321
Epoch:1 , Batch:1000 , Loss:0.0280
Epoch:2 , Batch:100 , Loss:0.0278
Epoch:2 , Batch:200 , Loss:0.0247
Epoch:2 , Batch:300 , Loss:0.0269
Epoch:2 , Batch:400 , Loss:0.0274
Epoch:2 , Batch:500 , Loss:0.0297
Epoch:2 , Batch:600 , Loss:0.0270
Epoch:2 , Batch:700 , Loss:0.0232
Epoch:2 , Batch:800 , Loss:0.0283
Epoch:2 , Batch:900 , Loss:0.0270
Epoch:2 , Batch:1000 , Loss:0.0244
Epoch:3 , Batch:100 , Loss:0.0242
Epoch:3 , Batch:200 , Loss:0.0243
Epoch:3 , Batch:300 , Loss:0.0233
Epoch:3 , Batch:400 , Loss:0.0199
Epoch:3 , Batch:500 , Loss:0.0223
Epoch:3 , Batch:600 , Loss:0.0209
Epoch:3 , Batch:700 , Loss:0.0209
Epoch:3 , Batch:800 , Loss:0.0180
Epoch:3 , Batch:900 , Loss:0.0202
Epoch:3 , Ba

In [18]:
correct_label=0
total_label=0

net.eval()
with torch.no_grad():
  for image,label in test1_loader:
    output=net(image)
    _,predicted=torch.max(output,1)
    total_label+=label.size(0)
    correct_label+=(predicted==label).sum().item()

In [19]:
print(f"Accuracy is {100*correct_label/total_label:.2f}")

Accuracy is 94.15
